## Imports

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset


## 1. Chargement des données de référence

In [ ]:
chemin_reference = "../chemin/vers/fichier_entrainement.csv"
reference_data = pd.read_csv(chemin_reference)


## 2. Chargement des données de production (logs de l'API)

In [ ]:
DATABASE_URL = os.getenv("DATABASE_URL")

if DATABASE_URL and DATABASE_URL.startswith("postgres://"):
    DATABASE_URL = DATABASE_URL.replace("postgres://", "postgresql://", 1)

if not DATABASE_URL:
    raise ValueError("La variable DATABASE_URL n'est pas configurée dans le fichier .env")

engine = create_engine(DATABASE_URL)
query = "SELECT input_data FROM prediction_logs WHERE status = 'success'"
df_logs = pd.read_sql(query, engine)

if df_logs.empty:
    print("La base de données ne contient aucun log d'inférence en succès.")
    production_data = pd.DataFrame(columns=reference_data.columns)
else:
    production_data = pd.json_normalize(df_logs['input_data'])
    print(f"Données de production chargées depuis Neon : {production_data.shape[0]} lignes.")


## 3. Alignement des colonnes
### Evidently compare les colonnes communes. On exclure les métadonnées de l'API.


In [ ]:
colonnes_communes = [col for col in production_data.columns if col in reference_data.columns]

reference_data_aligned = reference_data[colonnes_communes]
production_data_aligned = production_data[colonnes_communes]

## 4. Initialisation du rapport avec les métriques de Data Drift

In [ ]:
drift_report = Report(metrics=[DataDriftPreset()])



## 5. Calcul des statistiques comparatives (Référence vs Production)


In [ ]:
if not production_data_aligned.empty:
    # Lancement des calculs
    drift_report.run(reference_data=reference_data_aligned, current_data=production_data_aligned)
    
    # Affichage du rapport directement sous la cellule
    drift_report.show(mode='inline')
else:
    print("Calcul impossible : le jeu de données de production est vide.")


## 6. Sauvegarde du rapport interactif


In [ ]:
if not production_data_aligned.empty:
    chemin_rapport = "data_drift_report.html"
    drift_report.save_html(chemin_rapport)
    print(f"Rapport généré avec succès et sauvegardé sous : {chemin_rapport}")
else:
    print("Aucun rapport à sauvegarder.")